# Coefficient B11

SS final-interval case: `i=N-1, j=N (and transpose)`, with $n=N-1$. This is a new derivation in the author’s coefficient layout, using **p** for normalized power. It replaces the corresponding final-interval placement; earlier intervals remain unchanged. SN transposes NS.

## Definition

Set $\chi_1=\chi_{N-1}$, $\chi_2=\chi_N$ and $\Delta\chi=\chi_2-\chi_1$.
The density hats are $L(\chi)=(\chi_2-\chi)/\Delta\chi$ and $R(\chi)=(\chi-\chi_1)/\Delta\chi$.
For $\chi_1\leq\chi\leq\chi_2$, the source lower limit is the evaluation point:

$$F(\chi)=\int_\chi^{\chi_2}\frac{\chi_2-u}{\Delta\chi}\frac{u-\chi}{u}\,du,
\qquad H(\chi)=\int_\chi^{\chi_2}\frac{u-\chi_1}{\Delta\chi}\frac{u-\chi}{u}\,du.$$

There is no source half beyond $\chi_N$. Below $\chi_1$, the rising source instead has its full support $[\chi_1,\chi_2]$. At $\chi_1>0$ the value and first derivative match; both partial source integrals vanish at $\chi_2$.


## Change of variables

$$t=\frac{\chi}{\chi_2},\quad a=1-\frac{\chi_1}{\chi_2},\quad
p=1-\frac{P_1}{P_2},\quad z=1-\frac{1+z_1}{1+z_2}.$$

For an ordinary interval $0<a<1$, $1-a\leq t\leq1$, and $p,z$ are real. The normalized power notation presumes $P_2\neq0$; an endpoint-linear equivalent below removes that restriction. $P_1,P_2$ may be signed effective component power.

$$Q(t)=1-\frac{p}{a}(1-t),\qquad Z(t)=1-\frac{z}{a}(1-t).$$
The integrated source functions are $F(\chi)=\chi_2F_a(t)$ and $H(\chi)=\chi_2H_a(t)$:

$$F_a(t)=\frac{1-t^2}{2a}+\frac{t\log t}{a},$$
$$H_a(t)=\frac{(1-t)^2}{2a}-\frac{(1-a)(1-t)}a-\frac{(1-a)t\log t}a.$$
The continuous endpoint value is $t\log t\to0$ as $t\to0^+$. Physical lensing amplitudes are applied by the component’s existing factor/effective-power convention, exactly once; these radial basis coefficients introduce no additional amplitude.


In [1]:
import sympy as sp
from IPython.display import display

t, a, p, z, b = sp.symbols('t a p z b', real=True)
Q = 1-p*(1-t)/a
Z = 1-z*(1-t)/a
F = ((1-t*t)/2+t*sp.log(t))/a
H = ((1-t)**2/2-(1-a)*(1-t)-(1-a)*t*sp.log(t))/a
u = sp.symbols('u', positive=True)
# Moving-limit source integration, expressed in the same normalized units.
F_PRIMITIVE = sp.integrate((1-u)*(u-t)/(a*u), u)
H_PRIMITIVE = sp.integrate((u-1+a)*(u-t)/(a*u), u)
assert sp.simplify(F_PRIMITIVE.subs(u,1)-F_PRIMITIVE.subs(u,t)-F) == 0
assert sp.simplify(H_PRIMITIVE.subs(u,1)-H_PRIMITIVE.subs(u,t)-H) == 0
display(F, H)


(-t**2/2 + t*log(t) + 1/2)/a

(-t*(1 - a)*log(t) - (1 - a)*(1 - t) + (1 - t)**2/2)/a

## Ordinary coefficient $I_{11}$

$$I_{11}=\int_{1-a}^1 \frac{\left(1 - \frac{p \left(1 - t\right)}{a}\right) \left(1 - \frac{z \left(1 - t\right)}{a}\right)^{2} \left(- \frac{t^{2}}{2} + t \log{\left(t \right)} + \frac{1}{2}\right) \left(- t \left(1 - a\right) \log{\left(t \right)} - \left(1 - a\right) \left(1 - t\right) + \frac{\left(1 - t\right)^{2}}{2}\right)}{a^{2}}\,dt.$$

The dimensional coefficient is $B_{11}=\chi_2^3 P_2(1+z_2)^2 I_{11}$. Expand into powers of $t$ and $\log t$, then integrate each moment. For $k\neq-1$, the primitive of $t^k(\log t)^r$ is obtained by integration by parts; for $k=-1$, use $(\log t)^{r+1}/(r+1)$.

In [2]:
def integrate_moments(expression, lower):
    """Integrate every t**n log(t)**r term exactly on [lower, 1]."""
    result = 0
    for term in sp.Add.make_args(sp.expand(expression)):
        powers = term.as_powers_dict()
        n = int(powers.get(t, 0))
        r = int(powers.get(sp.log(t), 0))
        coefficient = term / t**n / sp.log(t)**r
        if n == -1:
            result -= coefficient*sp.log(lower)**(r+1)/sp.Integer(r+1)
        else:
            k = sp.Integer(n+1)
            primitive = t**k*sum((-1)**j*sp.factorial(r)/sp.factorial(r-j)
                                *sp.log(t)**(r-j)/k**(j+1) for j in range(r+1))
            boundary = 0 if lower == 0 else primitive.subs(t, lower)
            result += coefficient*(primitive.subs(t, 1)-boundary)
    return sp.factor(result)

INTEGRAND = F*H*Q*Z**2
EXPANDED = sp.expand(INTEGRAND)
I11 = integrate_moments(EXPANDED, 1-a)
display(EXPANDED, I11)


-t**3*log(t)/(2*a) + t**3/(2*a) + t**2*log(t)**2/a - t**2*log(t)/a - t**2/(2*a) + 3*t*log(t)/(2*a) - t/(2*a) + 1/(2*a) - p*t**4*log(t)/(2*a**2) + p*t**4/(2*a**2) + p*t**3*log(t)**2/a**2 - p*t**3*log(t)/(2*a**2) - p*t**3/a**2 - p*t**2*log(t)**2/a**2 + 5*p*t**2*log(t)/(2*a**2) - 3*p*t*log(t)/(2*a**2) + p*t/a**2 - p/(2*a**2) - t**4*z*log(t)/a**2 + t**4*z/a**2 - t**4/(4*a**2) + 2*t**3*z*log(t)**2/a**2 - t**3*z*log(t)/a**2 - 2*t**3*z/a**2 + t**3*log(t)/a**2 - 2*t**2*z*log(t)**2/a**2 + 5*t**2*z*log(t)/a**2 - t**2*log(t)**2/a**2 + t**2/(2*a**2) - 3*t*z*log(t)/a**2 + 2*t*z/a**2 - t*log(t)/a**2 - z/a**2 - 1/(4*a**2) - p*t**5*z*log(t)/a**3 + p*t**5*z/a**3 - p*t**5/(4*a**3) + 2*p*t**4*z*log(t)**2/a**3 - 3*p*t**4*z/a**3 + p*t**4*log(t)/a**3 + p*t**4/(4*a**3) - 4*p*t**3*z*log(t)**2/a**3 + 6*p*t**3*z*log(t)/a**3 + 2*p*t**3*z/a**3 - p*t**3*log(t)**2/a**3 - p*t**3*log(t)/a**3 + p*t**3/(2*a**3) + 2*p*t**2*z*log(t)**2/a**3 - 8*p*t**2*z*log(t)/a**3 + 2*p*t**2*z/a**3 + p*t**2*log(t)**2/a**3 - p*t**2*log(t

-(378000*a**8*p*z**2*log(1 - a) - 597375*a**8*p*z**2 - 882000*a**8*p*z*log(1 - a) + 1407000*a**8*p*z + 529200*a**8*p*log(1 - a) - 855540*a**8*p - 441000*a**8*z**2*log(1 - a) + 703500*a**8*z**2 + 1058400*a**8*z*log(1 - a) - 1711080*a**8*z - 661500*a**8*log(1 - a) + 1091475*a**8 + 882000*a**7*p*z**2*log(1 - a)**2 - 3255000*a**7*p*z**2*log(1 - a) + 2099500*a**7*p*z**2 - 2116800*a**7*p*z*log(1 - a)**2 + 7902720*a**7*p*z*log(1 - a) - 5226144*a**7*p*z + 1323000*a**7*p*log(1 - a)**2 - 5027400*a**7*p*log(1 - a) + 3453030*a**7*p - 1058400*a**7*z**2*log(1 - a)**2 + 3951360*a**7*z**2*log(1 - a) - 2613072*a**7*z**2 + 2646000*a**7*z*log(1 - a)**2 - 10054800*a**7*z*log(1 - a) + 6906060*a**7*z - 1764000*a**7*log(1 - a)**2 + 6909000*a**7*log(1 - a) - 5059250*a**7 - 2998800*a**6*p*z**2*log(1 - a)**2 + 5550720*a**6*p*z**2*log(1 - a) - 1343944*a**6*p*z**2 + 7408800*a**6*p*z*log(1 - a)**2 - 14076720*a**6*p*z*log(1 - a) + 3609144*a**6*p*z - 4851000*a**6*p*log(1 - a)**2 + 9628500*a**6*p*log(1 - a) - 2735425

### Result

$$I_{11}=- \frac{378000 a^{8} p z^{2} \log{\left(1 - a \right)} - 597375 a^{8} p z^{2} - 882000 a^{8} p z \log{\left(1 - a \right)} + 1407000 a^{8} p z + 529200 a^{8} p \log{\left(1 - a \right)} - 855540 a^{8} p - 441000 a^{8} z^{2} \log{\left(1 - a \right)} + 703500 a^{8} z^{2} + 1058400 a^{8} z \log{\left(1 - a \right)} - 1711080 a^{8} z - 661500 a^{8} \log{\left(1 - a \right)} + 1091475 a^{8} + 882000 a^{7} p z^{2} \log{\left(1 - a \right)}^{2} - 3255000 a^{7} p z^{2} \log{\left(1 - a \right)} + 2099500 a^{7} p z^{2} - 2116800 a^{7} p z \log{\left(1 - a \right)}^{2} + 7902720 a^{7} p z \log{\left(1 - a \right)} - 5226144 a^{7} p z + 1323000 a^{7} p \log{\left(1 - a \right)}^{2} - 5027400 a^{7} p \log{\left(1 - a \right)} + 3453030 a^{7} p - 1058400 a^{7} z^{2} \log{\left(1 - a \right)}^{2} + 3951360 a^{7} z^{2} \log{\left(1 - a \right)} - 2613072 a^{7} z^{2} + 2646000 a^{7} z \log{\left(1 - a \right)}^{2} - 10054800 a^{7} z \log{\left(1 - a \right)} + 6906060 a^{7} z - 1764000 a^{7} \log{\left(1 - a \right)}^{2} + 6909000 a^{7} \log{\left(1 - a \right)} - 5059250 a^{7} - 2998800 a^{6} p z^{2} \log{\left(1 - a \right)}^{2} + 5550720 a^{6} p z^{2} \log{\left(1 - a \right)} - 1343944 a^{6} p z^{2} + 7408800 a^{6} p z \log{\left(1 - a \right)}^{2} - 14076720 a^{6} p z \log{\left(1 - a \right)} + 3609144 a^{6} p z - 4851000 a^{6} p \log{\left(1 - a \right)}^{2} + 9628500 a^{6} p \log{\left(1 - a \right)} - 2735425 a^{6} p + 3704400 a^{6} z^{2} \log{\left(1 - a \right)}^{2} - 7038360 a^{6} z^{2} \log{\left(1 - a \right)} + 1804572 a^{6} z^{2} - 9702000 a^{6} z \log{\left(1 - a \right)}^{2} + 19257000 a^{6} z \log{\left(1 - a \right)} - 5470850 a^{6} z + 7056000 a^{6} \log{\left(1 - a \right)}^{2} - 15288000 a^{6} \log{\left(1 - a \right)} + 5426750 a^{6} + 3439800 a^{5} p z^{2} \log{\left(1 - a \right)}^{2} - 2654820 a^{5} p z^{2} \log{\left(1 - a \right)} - 26061 a^{5} p z^{2} - 8820000 a^{5} p z \log{\left(1 - a \right)}^{2} + 7114800 a^{5} p z \log{\left(1 - a \right)} + 100940 a^{5} p z + 6174000 a^{5} p \log{\left(1 - a \right)}^{2} - 5439000 a^{5} p \log{\left(1 - a \right)} - 83300 a^{5} p - 4410000 a^{5} z^{2} \log{\left(1 - a \right)}^{2} + 3557400 a^{5} z^{2} \log{\left(1 - a \right)} + 50470 a^{5} z^{2} + 12348000 a^{5} z \log{\left(1 - a \right)}^{2} - 10878000 a^{5} z \log{\left(1 - a \right)} - 166600 a^{5} z - 10584000 a^{5} \log{\left(1 - a \right)}^{2} + 12348000 a^{5} \log{\left(1 - a \right)} - 1470000 a^{5} - 1323000 a^{4} p z^{2} \log{\left(1 - a \right)}^{2} - 14700 a^{4} p z^{2} \log{\left(1 - a \right)} - 33285 a^{4} p z^{2} + 3528000 a^{4} p z \log{\left(1 - a \right)}^{2} + 117600 a^{4} p z \log{\left(1 - a \right)} + 113680 a^{4} p z - 2646000 a^{4} p \log{\left(1 - a \right)}^{2} - 441000 a^{4} p \log{\left(1 - a \right)} + 257250 a^{4} p + 1764000 a^{4} z^{2} \log{\left(1 - a \right)}^{2} + 58800 a^{4} z^{2} \log{\left(1 - a \right)} + 56840 a^{4} z^{2} - 5292000 a^{4} z \log{\left(1 - a \right)}^{2} - 882000 a^{4} z \log{\left(1 - a \right)} + 514500 a^{4} z + 7056000 a^{4} \log{\left(1 - a \right)}^{2} - 3160500 a^{4} \log{\left(1 - a \right)} - 147000 a^{4} - 29400 a^{3} p z^{2} \log{\left(1 - a \right)} - 41370 a^{3} p z^{2} + 352800 a^{3} p z \log{\left(1 - a \right)} - 99960 a^{3} p z - 441000 a^{3} p \log{\left(1 - a \right)}^{2} + 646800 a^{3} p \log{\left(1 - a \right)} + 102900 a^{3} p + 176400 a^{3} z^{2} \log{\left(1 - a \right)} - 49980 a^{3} z^{2} - 882000 a^{3} z \log{\left(1 - a \right)}^{2} + 1293600 a^{3} z \log{\left(1 - a \right)} + 205800 a^{3} z - 1764000 a^{3} \log{\left(1 - a \right)}^{2} - 147000 a^{3} \log{\left(1 - a \right)} - 88200 a^{2} p z^{2} \log{\left(1 - a \right)} - 4410 a^{2} p z^{2} + 352800 a^{2} p z \log{\left(1 - a \right)}^{2} - 376320 a^{2} p z \log{\left(1 - a \right)} - 152880 a^{2} p z + 441000 a^{2} p \log{\left(1 - a \right)}^{2} + 102900 a^{2} p \log{\left(1 - a \right)} + 176400 a^{2} z^{2} \log{\left(1 - a \right)}^{2} - 188160 a^{2} z^{2} \log{\left(1 - a \right)} - 76440 a^{2} z^{2} + 882000 a^{2} z \log{\left(1 - a \right)}^{2} + 205800 a^{2} z \log{\left(1 - a \right)} - 88200 a p z^{2} \log{\left(1 - a \right)}^{2} + 54180 a p z^{2} \log{\left(1 - a \right)} + 59220 a p z^{2} - 352800 a p z \log{\left(1 - a \right)}^{2} - 152880 a p z \log{\left(1 - a \right)} - 176400 a z^{2} \log{\left(1 - a \right)}^{2} - 76440 a z^{2} \log{\left(1 - a \right)} + 88200 p z^{2} \log{\left(1 - a \right)}^{2} + 59220 p z^{2} \log{\left(1 - a \right)}}{5292000 a^{5}}.$$

## Observer coefficient $J_{11}$

Set $a=1$, but replace the power with $Q(t)=t^3$ **before** integration. Substituting $p=1$ into $I_{11}$ would preserve linear power and is not this observer convention. The dimensional prefactor is the same.

In [3]:
OBSERVER_INTEGRAND = sp.cancel(INTEGRAND/Q).subs(a,1)*t**3
J11 = integrate_moments(OBSERVER_INTEGRAND, 0)
display(OBSERVER_INTEGRAND, J11)


t**3*(-t**6*z**2/4 + t**5*z**2*log(t)/2 + t**5*z**2 - t**5*z/2 - 2*t**4*z**2*log(t) - 5*t**4*z**2/4 + t**4*z*log(t) + 3*t**4*z/2 - t**4/4 + 3*t**3*z**2*log(t) - 3*t**3*z*log(t) - t**3*z + t**3*log(t)/2 + t**3/2 - 2*t**2*z**2*log(t) + 5*t**2*z**2/4 + 3*t**2*z*log(t) - t**2*z - t**2*log(t) + t*z**2*log(t)/2 - t*z**2 - t*z*log(t) + 3*t*z/2 + t*log(t)/2 - t/2 + z**2/4 - z/2 + 1/4)

(652*z**2 - 1989*z + 1602)/6350400

### Result

$$J_{11}=\frac{163 z^{2}}{1587600} - \frac{221 z}{705600} + \frac{89}{352800}.$$

## Correspondence with the existing interior source

The falling half after the source node has width $b=(\chi_{n+2}-\chi_{n+1})/\chi_{n+1}$. Its full-support contribution is $D(t,b)=b/2+t[1-(1+b)\log(1+b)/b]$. Therefore $D\to0$ as $b\to0^+$, and the interior source $H+D$ becomes $H$. The following also verifies the integrated result against the original SS B02 text export, rather than directly substituting into a singular $1/b$ expression.

In [4]:
import re
from pathlib import Path

D = b/2+t*(1-(1+b)*sp.log(1+b)/b)
assert sp.limit(D,b,0,dir='+') == 0
INTERIOR_INTEGRAND = F*(H+D)*Q*Z**2
assert sp.simplify(sp.limit(INTERIOR_INTEGRAND,b,0,dir='+')-INTEGRAND) == 0
FILENAME = 'Coefficient_B02.txt'
HERE = Path.cwd()
for CANDIDATE in (HERE, HERE/'notebooks'/'derivation'/'SS'):
    if (CANDIDATE/FILENAME).is_file():
        HERE = CANDIDATE
        break
ORIGINAL = (HERE/FILENAME).read_text()
PARTS = re.split(r'\b[IJ]\d+\s*=', ORIGINAL)[1:]
for TEXT, TARGET in zip(PARTS, (I11,J11), strict=True):
    EXPR = sp.sympify(TEXT.replace('numpy.log','log').replace('\n',' '), locals={'a':a,'b':b,'p':p,'z':z,'log':sp.log})
    # Denominator degree is <=2: terms through b^4 preserve its finite limit.
    REGULAR = sp.cancel(EXPR.subs(sp.log(1+b), b-b**2/2+b**3/3-b**4/4))
    assert sp.factor(REGULAR.subs(b,0)-TARGET) == 0
print('Both I and J agree with the original interior-source limit.')


Both I and J agree with the original interior-source limit.


## Stable evaluation and zero endpoint powers

Use $s=(1-t)/a\in[0,1]$ and $d_m=1/[(m+2)(m+3)]$. Define
$$f(v)=\sum_{m=0}^{\infty}d_m v^m,\qquad
g(v)=\sum_{m=0}^{\infty}\left(\frac12-\frac1{m+3}\right)v^m.$$
Then
$$F_a=a^2s^3f(as),\quad H_a=\frac{a^2s^2}2-(1-a)F_a,$$
$$\frac{F_a}{t}=a^2s^3g(as),\quad
\frac{H_a}{t}=\frac{a^2s^2}{2(1-as)}-(1-a)\frac{F_a}{t}.$$
For SS, $f(v)^2=\sum e_m v^m$, with $e_m=\sum_{j=0}^m d_jd_{m-j}$; multiply the two source expressions before integrating.

All remaining factors are polynomials in $s$:
$$P(s)=P_2+(P_1-P_2)s,\qquad (1+z)(s)=(1+z_2)+(z_1-z_2)s.$$
Integrate each resulting monomial using $\int_0^1s^qds=1/(q+1)$. This gives analytic series of **integrated moments**, with no production quadrature. The NS scale is $\chi_2a^3$, and the SS scale is $\chi_2^3a^5$ when endpoint powers/redshifts are included in the moments.

Production uses 16 terms for $a\le1/16$, 32 for $a\le1/4$, 64 for $a\le1/2$, and 160 for $a\le3/4$. The geometric tail parameter $a^K$ is below $6\times10^{-20}$ in each regime. The coefficients and moment factors are bounded, with at most a factor $1/(1-a)\le4$ from the geometric tail. Broader intervals use the displayed elementary closed form; the observer uses its separate $J$ expression. Floating-point cancellation from an intentionally sign-changing final spectrum is assessed with absolute error too.

The direct endpoint-linear moments avoid $P_1/P_2$. For the closed form, $I=A+Bp$ implies
$$P_2I=-BP_1+(A+B)P_2.$$
This equivalence handles either endpoint being zero without changing a signed contribution or introducing an epsilon.


In [5]:
LEFT_WEIGHT = sp.factor(-sp.diff(I11,p))
RIGHT_WEIGHT = sp.factor(I11.subs(p,0)-LEFT_WEIGHT)
assert sp.simplify(I11-(LEFT_WEIGHT*(1-p)+RIGHT_WEIGHT)) == 0
display(LEFT_WEIGHT, RIGHT_WEIGHT)


(378000*a**8*z**2*log(1 - a) - 597375*a**8*z**2 - 882000*a**8*z*log(1 - a) + 1407000*a**8*z + 529200*a**8*log(1 - a) - 855540*a**8 + 882000*a**7*z**2*log(1 - a)**2 - 3255000*a**7*z**2*log(1 - a) + 2099500*a**7*z**2 - 2116800*a**7*z*log(1 - a)**2 + 7902720*a**7*z*log(1 - a) - 5226144*a**7*z + 1323000*a**7*log(1 - a)**2 - 5027400*a**7*log(1 - a) + 3453030*a**7 - 2998800*a**6*z**2*log(1 - a)**2 + 5550720*a**6*z**2*log(1 - a) - 1343944*a**6*z**2 + 7408800*a**6*z*log(1 - a)**2 - 14076720*a**6*z*log(1 - a) + 3609144*a**6*z - 4851000*a**6*log(1 - a)**2 + 9628500*a**6*log(1 - a) - 2735425*a**6 + 3439800*a**5*z**2*log(1 - a)**2 - 2654820*a**5*z**2*log(1 - a) - 26061*a**5*z**2 - 8820000*a**5*z*log(1 - a)**2 + 7114800*a**5*z*log(1 - a) + 100940*a**5*z + 6174000*a**5*log(1 - a)**2 - 5439000*a**5*log(1 - a) - 83300*a**5 - 1323000*a**4*z**2*log(1 - a)**2 - 14700*a**4*z**2*log(1 - a) - 33285*a**4*z**2 + 3528000*a**4*z*log(1 - a)**2 + 117600*a**4*z*log(1 - a) + 113680*a**4*z - 2646000*a**4*log(1 - a)*

(63000*a**8*z**2*log(1 - a) - 106125*a**8*z**2 - 176400*a**8*z*log(1 - a) + 304080*a**8*z + 132300*a**8*log(1 - a) - 235935*a**8 + 176400*a**7*z**2*log(1 - a)**2 - 696360*a**7*z**2*log(1 - a) + 513572*a**7*z**2 - 529200*a**7*z*log(1 - a)**2 + 2152080*a**7*z*log(1 - a) - 1679916*a**7*z + 441000*a**7*log(1 - a)**2 - 1881600*a**7*log(1 - a) + 1606220*a**7 - 705600*a**6*z**2*log(1 - a)**2 + 1487640*a**6*z**2*log(1 - a) - 460628*a**6*z**2 + 2293200*a**6*z*log(1 - a)**2 - 5180280*a**6*z*log(1 - a) + 1861706*a**6*z - 2205000*a**6*log(1 - a)**2 + 5659500*a**6*log(1 - a) - 2691325*a**6 + 970200*a**5*z**2*log(1 - a)**2 - 902580*a**5*z**2*log(1 - a) - 24409*a**5*z**2 - 3528000*a**5*z*log(1 - a)**2 + 3763200*a**5*z*log(1 - a) + 65660*a**5*z + 4410000*a**5*log(1 - a)**2 - 6909000*a**5*log(1 - a) + 1553300*a**5 - 441000*a**4*z**2*log(1 - a)**2 - 44100*a**4*z**2*log(1 - a) - 23555*a**4*z**2 + 1764000*a**4*z*log(1 - a)**2 + 764400*a**4*z*log(1 - a) - 628180*a**4*z - 4410000*a**4*log(1 - a)**2 + 360150

## Export and validation

[Coefficient_B11.txt](Coefficient_B11.txt) contains the ordinary and observer expressions in the existing `I11 = ...`, `J11 = ...` format. [Coefficient_B11_Validation.ipynb](Coefficient_B11_Validation.ipynb) compares the compiled float64 coefficient against independent integration of the original hats and moving source limits. The original Mathematica derivations remain unchanged.